In [0]:
spark.conf.set(
  "fs.azure.account.key.adprojectbradleydev.blob.core.windows.net",
  "Azure key"
)


In [0]:
from pyspark.sql.functions import col, round
from pyspark.sql import SparkSession

# Load from bronze container
df = spark.read.option("header", True).csv("wasbs://bronze@adprojectbradleydev.blob.core.windows.net/facebook_ads.csv")

# Clean up data
df_clean = df.withColumn("impressions", col("impressions").cast("int")) \
             .withColumn("clicks", col("clicks").cast("int")) \
             .withColumn("spent", col("spent").cast("float")) \
             .withColumn("CTR", round(col("clicks") / col("impressions"), 4)) \
             .withColumn("CPC", round(col("spent") / col("clicks"), 2)) \

df_clean.show(5)


+------+---------------+-------------+-----------+--------------+-----+------+---------+---------+---------+-----------+------+-----+----------------+-------------------+------+----+
| ad_id|reporting_start|reporting_end|campaign_id|fb_campaign_id|  age|gender|interest1|interest2|interest3|impressions|clicks|spent|total_conversion|approved_conversion|   CTR| CPC|
+------+---------------+-------------+-----------+--------------+-----+------+---------+---------+---------+-----------+------+-----+----------------+-------------------+------+----+
|708746|     17/08/2017|   17/08/2017|        916|        103916|30-34|     M|       15|       17|       17|       7350|     1| 1.43|               2|                  1|1.0E-4|1.43|
|708749|     17/08/2017|   17/08/2017|        916|        103917|30-34|     M|       16|       19|       21|      17861|     2| 1.82|               2|                  0|1.0E-4|0.91|
|708771|     17/08/2017|   17/08/2017|        916|        103920|30-34|     M|       

In [0]:
df_clean.write.mode("overwrite").option("header", True).csv("wasbs://silver@adprojectbradleydev.blob.core.windows.net/cleaned_facebook_ads.csv")


In [0]:
df_gold = df_clean.groupBy("campaign_id").agg(
    {"impressions": "sum", "clicks": "sum", "spent": "sum"}
).withColumnRenamed("sum(impressions)", "total_impressions") \
 .withColumnRenamed("sum(clicks)", "total_clicks") \
 .withColumnRenamed("sum(spent)", "total_spend") \
 .withColumn("CTR", round(col("total_clicks") / col("total_impressions"), 4)) \
 .withColumn("CPC", round(col("total_spend") / col("total_clicks"), 2))

df_gold.write.mode("overwrite").option("header", True).csv("wasbs://gold@adprojectbradleydev.blob.core.windows.net/summary_ads.csv")
